<a href="https://colab.research.google.com/github/vlotran/HW2-Excellent-Bigframes-PyTorch/blob/main/task1_bigframes_embeddings_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Authentication
Authenticate to Google Cloud (needed for BigQuery/Bigframes).

In [2]:
from google.colab import auth
auth.authenticate_user()

 # Project + Region
 Set the GCP project and BigQuery location used by Bigframes.

In [3]:
PROJECT_ID = "hw2-excellent"
REGION = "US"

!gcloud config set project {PROJECT_ID}

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [22]:
!gcloud services enable bigqueryconnection.googleapis.com --project hw2-excellent

# Install + Bigframes sanity check
Install dependencies and verify Bigframes can run a trivial BigQuery query.

In [4]:
!pip -q install bigframes google-cloud-bigquery

import bigframes.pandas as bf
bf.options.bigquery.project = PROJECT_ID
bf.options.bigquery.location = REGION

bf.read_gbq("SELECT 1 AS ok").to_pandas()

,ok
0,1


# Sanity check: BigQuery Embeddings (Bigframes)

In [11]:
from bigframes.ml.llm import TextEmbeddingGenerator

emb = TextEmbeddingGenerator()  # uses BigQuery/Vertex embedding endpoint
test = bf.DataFrame({"text": ["hello world", "bigquery is great"]})

out = emb.predict(test)
out.head().to_pandas()

,ml_generate_embedding_result,ml_generate_embedding_statistics,ml_generate_embedding_status,content
0,[ 2.33469512e-02 -1.58897862e-02 -3.47731076e-...,"{""token_count"":2,""truncated"":false}",,hello world
1,[-1.74164660e-02 -1.08140586e-02 -5.22318147e-...,"{""token_count"":5,""truncated"":false}",,bigquery is great


# Sanity check: BigQuery LLM Text Generation (Bigframes)

In [23]:
import bigframes.pandas
from bigframes.ml.llm import GeminiTextGenerator

session = bigframes.pandas.get_global_session()
connection = f"{PROJECT_ID}.us.bigframes-default-connection"

gen = GeminiTextGenerator(
    session=session,
    connection_name=connection,
    model_name="gemini-1.5-flash-001",
)

prompt_df = bf.DataFrame({"prompt": ["Explain BigQuery in one sentence."]})
gen.predict(prompt_df).head().to_pandas()

BadRequest: 400 Not found: Publisher Model `projects/hw2-excellent/locations/us-central1/publishers/google/models/gemini-1.5-flash-001` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions; reason: invalidQuery, location: query, message: Not found: Publisher Model `projects/hw2-excellent/locations/us-central1/publishers/google/models/gemini-1.5-flash-001` was not found or your project does not have access to it. Please ensure you are using a valid model version. For more information, see: https://cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

Location: US
Job ID: 01c2d6e6-cf80-455b-891e-2a0f118e9603


In [21]:
!gcloud services list --enabled --project hw2-excellent | grep aiplatform

aiplatform.googleapis.com            Vertex AI API
